# 🟠 BrightLearn — Advanced SQL Practical Notebook

**Topics covered:** Wildcards · Window Functions · DDL · DML · Advanced Querying · Views · Stored Procedures

**How to use this notebook:**
- Read each explanation cell carefully before running anything.
- Run the *worked example* cell to see the output.
- Then complete the *YOUR TURN* cell yourself — write your own SQL from scratch.
- Do not skip the YOUR TURN cells. That is where the learning happens.

**Schema:** `brightcoffee` — run the setup cell below before anything else.

---
| Table | What it contains |
|-------|-----------------|
| `dim_customers` | Customer profiles — id, name, email, tier, loyalty_points |
| `dim_products` | Product catalogue — id, code, name, category, price |
| `dim_stores` | Store locations — id, name, city, region |
| `dim_date` | Date dimension — date_id, full_date, year, month, quarter |
| `fact_orders` | Sales transactions — order_id, customer_id, product_id, store_id, date, amounts |
| `customer_spend_summary` | Pre-built summary: customer_id, first_name, tier, total_orders, total_spent |


##  Setup — Run Every Cell in This Section First

This section creates the `brightcoffee` schema and all tables, then loads
sample data so every section of the notebook has real rows to work with.

**Run order:** top to bottom, one cell at a time.
If you re-run the notebook from scratch, re-run this section first.

| Step | What it does |
|------|-------------|
| 1 | Create schema `brightcoffee` |
| 2 | Create `dim_customers` |
| 3 | Create `dim_products` |
| 4 | Create `dim_stores` |
| 5 | Create `dim_date` |
| 6 | Create `fact_orders` |
| 7 | Load sample data into all tables |
| 8 | Build `customer_spend_summary` helper table |
| 9 | Confirm row counts |


In [0]:
-- ─────────────────────────────────────────────────────────────
-- STEP 1: Create and activate the schema
-- ─────────────────────────────────────────────────────────────
CREATE SCHEMA IF NOT EXISTS brightcoffee
    COMMENT 'BrightLearn — Bright Coffee Shop analytics schema';

USE brightcoffee;

SELECT current_database();  -- expected: brightcoffee


In [0]:
-- ─────────────────────────────────────────────────────────────
-- STEP 2: dim_customers
-- NOT NULL and DEFAULT inline; UNIQUE and CHECK added after
-- ─────────────────────────────────────────────────────────────
DROP TABLE IF EXISTS dim_customers;

CREATE TABLE dim_customers (
    customer_id    BIGINT  NOT NULL,
    first_name     STRING  NOT NULL,
    last_name      STRING  NOT NULL,
    email          STRING,
    tier           STRING  DEFAULT 'Bronze',
    loyalty_points INT     DEFAULT 0,
    created_at     DATE,
    CONSTRAINT pk_customer PRIMARY KEY (customer_id)
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

alter table dim_customers 
set TBLPROPERTIES('spark.databricks.sql.dsv2.unique.enabled' =True);

-- Add UNIQUE and CHECK after creation
-- ALTER TABLE dim_customers
--     ADD CONSTRAINT uq_customer_email UNIQUE (email);

ALTER TABLE dim_customers
    ADD CONSTRAINT chk_customer_tier
    CHECK (tier IN ('Bronze', 'Silver', 'Gold'));

-- ALTER TABLE dim_customers
--     ALTER CONSTRAINT chk_customer_tier ENFORCED;

DESCRIBE TABLE dim_customers;


In [0]:
-- ─────────────────────────────────────────────────────────────
-- STEP 3: dim_products
-- ─────────────────────────────────────────────────────────────
DROP TABLE IF EXISTS dim_products;

CREATE TABLE dim_products (
    product_id    BIGINT         NOT NULL,
    product_code  STRING         NOT NULL,
    product_name  STRING         NOT NULL,
    price         DECIMAL(8,2)   NOT NULL,
    category      STRING         DEFAULT 'Uncategorized',
    is_active     BOOLEAN        DEFAULT true,
    CONSTRAINT pk_product PRIMARY KEY (product_id)
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

-- ALTER TABLE dim_products
--     ADD CONSTRAINT chk_product_price CHECK (price > 0);

-- ALTER TABLE dim_products
--     ADD CONSTRAINT uq_product_code UNIQUE (product_code);

-- DESCRIBE TABLE dim_products;


In [0]:
-- ─────────────────────────────────────────────────────────────
-- STEP 4: dim_stores
-- ─────────────────────────────────────────────────────────────
DROP TABLE IF EXISTS dim_stores;

CREATE TABLE dim_stores (
    store_id    BIGINT  NOT NULL,
    store_name  STRING  NOT NULL,
    location    STRING,
    city        STRING,
    region      STRING,
    CONSTRAINT pk_store PRIMARY KEY (store_id)
)
USING DELTA;

DESCRIBE TABLE dim_stores;


In [0]:
-- ─────────────────────────────────────────────────────────────
-- STEP 5: dim_date
-- ─────────────────────────────────────────────────────────────
DROP TABLE IF EXISTS dim_date;

CREATE TABLE dim_date (
    date_id      BIGINT  NOT NULL,
    full_date    DATE    NOT NULL,
    year         INT,
    month        INT,
    quarter      INT,
    day_of_week  STRING,
    CONSTRAINT pk_date PRIMARY KEY (date_id)
)
USING DELTA;

DESCRIBE TABLE dim_date;


In [0]:
-- ─────────────────────────────────────────────────────────────
-- STEP 6: fact_orders  (partitioned by order_date)
-- ─────────────────────────────────────────────────────────────
DROP TABLE IF EXISTS fact_orders;

CREATE TABLE fact_orders (
    order_id      BIGINT        NOT NULL,
    customer_id   BIGINT        NOT NULL,
    product_id    BIGINT        NOT NULL,
    store_id      BIGINT        NOT NULL,
    order_date    DATE          NOT NULL,
    quantity      INT           NOT NULL,
    unit_price    DECIMAL(10,2) NOT NULL,
    total_amount  DECIMAL(10,2) NOT NULL,
    CONSTRAINT pk_order PRIMARY KEY (order_id)
)
USING DELTA
PARTITIONED BY (order_date);

DESCRIBE TABLE fact_orders;


In [0]:
-- ─────────────────────────────────────────────────────────────
-- STEP 7a: Load dim_customers  (20 customers)
-- ─────────────────────────────────────────────────────────────
INSERT INTO dim_customers
    (customer_id, first_name, last_name, email, tier, loyalty_points, created_at)
VALUES
    (1,  'Amara',   'Nkosi',     'amara.nkosi@gmail.com',     'Gold',   520, '2022-03-15'),
    (2,  'Siya',    'Dlamini',   'siya.dlamini@yahoo.com',    'Silver', 210, '2022-05-20'),
    (3,  'Thandi',  'Mokoena',   'thandi.m@gmail.com',        'Bronze',  45, '2022-07-01'),
    (4,  'Lebo',    'Sithole',   'lebo.sithole@gmail.com',    'Gold',   480, '2022-08-14'),
    (5,  'Kagiso',  'Tau',       'kagiso.tau@outlook.com',    'Bronze',  30, '2022-09-10'),
    (6,  'Naledi',  'Khumalo',   'naledi.k@gmail.com',        'Silver', 175, '2022-10-05'),
    (7,  'Sipho',   'Zulu',      'sipho.zulu@yahoo.com',      'Gold',   610, '2022-11-22'),
    (8,  'Ayanda',  'Dube',      'ayanda.dube@gmail.com',     'Bronze',  15, '2023-01-08'),
    (9,  'Tebogo',  'Nkosi',     'tebogo.nkosi@outlook.com',  'Silver', 290, '2023-02-14'),
    (10, 'Refilwe', 'Mthembu',   'refilwe.m@gmail.com',       'Bronze',  60, '2023-03-19'),
    (11, 'Zanele',  'Khumalo',   'zanele.k@yahoo.com',        'Gold',   730, '2023-04-02'),
    (12, 'Mpho',    'Sithole',   'mpho.sithole@gmail.com',    'Silver', 155, '2023-05-17'),
    (13, 'Lindi',   'Mokoena',   'lindi.mokoena@gmail.com',   'Bronze',  25, '2023-06-23'),
    (14, 'Bongani', 'Dlamini',   'bongani.d@outlook.com',     'Silver', 320, '2023-07-30'),
    (15, 'Ntombi',  'Zulu',      'ntombi.zulu@gmail.com',     'Gold',   560, '2023-08-11'),
    (16, 'Karabo',  'Tau',       'karabo.tau@yahoo.com',      'Bronze',  10, '2023-09-04'),
    (17, 'Dineo',   'Nkosi',     'dineo.nkosi@gmail.com',     'Silver', 240, '2023-10-15'),
    (18, 'Lesego',  'Mthembu',   'lesego.m@outlook.com',      'Bronze',  55, '2023-11-28'),
    (19, 'Precious','Dube',      'precious.dube@gmail.com',   'Gold',   410, '2024-01-07'),
    (20, 'Tshepo',  'Khumalo',   'tshepo.k@yahoo.com',        'Silver', 190, '2024-02-19');

SELECT COUNT(*) AS customer_count FROM dim_customers;  -- expected: 20


In [0]:
-- ─────────────────────────────────────────────────────────────
-- STEP 7b: Load dim_products  (16 products)
-- ─────────────────────────────────────────────────────────────
INSERT INTO dim_products
    (product_id, product_code, product_name, price, category, is_active)
VALUES
    (1,  'FLAT001', 'Flat White',              38.00, 'Coffee',    true),
    (2,  'LATT001', 'Oat Latte',               42.00, 'Coffee',    true),
    (3,  'LATT002', 'Vanilla Latte',            44.00, 'Coffee',    true),
    (4,  'ESPR001', 'Double Espresso',          28.00, 'Coffee',    true),
    (5,  'CAPS001', 'Cappuccino',               36.00, 'Coffee',    true),
    (6,  'DECF001', 'Decaf Flat White',         38.00, 'Coffee',    true),
    (7,  'ICEC001', 'Iced Coffee',              40.00, 'Cold Brew', true),
    (8,  'CBRW001', 'Cold Brew Original',       45.00, 'Cold Brew', true),
    (9,  'CBRW002', 'Cold Brew Vanilla',        48.00, 'Cold Brew', true),
    (10, 'MATC001', 'Matcha Latte',             46.00, 'Tea',       true),
    (11, 'CHAI001', 'Chai Latte',               42.00, 'Tea',       true),
    (12, 'GRNT001', 'Green Tea',                32.00, 'Tea',       true),
    (13, 'MUFF001', 'Blueberry Muffin',         35.00, 'Food',      true),
    (14, 'CROI001', 'Butter Croissant',         30.00, 'Food',      true),
    (15, 'BNBR001', 'Banana Bread',             38.00, 'Food',      true),
    (16, 'DECF002', 'Decaf Oat Latte',          42.00, 'Coffee',    false);

SELECT COUNT(*) AS product_count FROM dim_products;  -- expected: 16


In [0]:
-- ─────────────────────────────────────────────────────────────
-- STEP 7c: Load dim_stores  (6 stores)
-- ─────────────────────────────────────────────────────────────
INSERT INTO dim_stores
    (store_id, store_name, location, city, region)
VALUES
    (1, 'Bright Coffee Sandton',    'Sandton City Mall',        'Johannesburg', 'North'),
    (2, 'Bright Coffee Rosebank',   'The Zone @ Rosebank',      'Johannesburg', 'North'),
    (3, 'Bright Coffee V&A',        'V&A Waterfront',           'Cape Town',    'West'),
    (4, 'Bright Coffee Stellenbosch','Eikestad Mall',           'Stellenbosch', 'West'),
    (5, 'Bright Coffee Umhlanga',   'Gateway Theatre of Shopping','Durban',     'East'),
    (6, 'Bright Coffee Pretoria',   'Menlyn Park',              'Pretoria',     'North');

SELECT COUNT(*) AS store_count FROM dim_stores;  -- expected: 6


In [0]:
-- ─────────────────────────────────────────────────────────────
-- STEP 7d: Load dim_date  (key dates used in fact_orders)
-- ─────────────────────────────────────────────────────────────
INSERT INTO dim_date
    (date_id, full_date, year, month, quarter, day_of_week)
VALUES
    (20240101, '2024-01-01', 2024, 1,  1, 'Monday'),
    (20240115, '2024-01-15', 2024, 1,  1, 'Monday'),
    (20240201, '2024-02-01', 2024, 2,  1, 'Thursday'),
    (20240214, '2024-02-14', 2024, 2,  1, 'Wednesday'),
    (20240301, '2024-03-01', 2024, 3,  1, 'Friday'),
    (20240315, '2024-03-15', 2024, 3,  1, 'Friday'),
    (20240401, '2024-04-01', 2024, 4,  2, 'Monday'),
    (20240415, '2024-04-15', 2024, 4,  2, 'Monday'),
    (20240501, '2024-05-01', 2024, 5,  2, 'Wednesday'),
    (20240515, '2024-05-15', 2024, 5,  2, 'Wednesday'),
    (20240601, '2024-06-01', 2024, 6,  2, 'Saturday'),
    (20240701, '2024-07-01', 2024, 7,  3, 'Monday'),
    (20240801, '2024-08-01', 2024, 8,  3, 'Thursday'),
    (20240901, '2024-09-01', 2024, 9,  3, 'Sunday'),
    (20241001, '2024-10-01', 2024, 10, 4, 'Tuesday'),
    (20241101, '2024-11-01', 2024, 11, 4, 'Friday'),
    (20241201, '2024-12-01', 2024, 12, 4, 'Sunday'),
    (20250101, '2025-01-01', 2025, 1,  1, 'Wednesday'),
    (20250201, '2025-02-01', 2025, 2,  1, 'Saturday'),
    (20250301, '2025-03-01', 2025, 3,  1, 'Saturday'),
    (20250401, '2025-04-01', 2025, 4,  2, 'Tuesday'),
    (20250501, '2025-05-01', 2025, 5,  2, 'Thursday');

SELECT COUNT(*) AS date_count FROM dim_date;  -- expected: 22


In [0]:
-- ─────────────────────────────────────────────────────────────
-- STEP 7e: Load fact_orders  (60 transactions)
-- ─────────────────────────────────────────────────────────────
INSERT INTO fact_orders
    (order_id, customer_id, product_id, store_id, order_date, quantity, unit_price, total_amount)
VALUES
    -- Customer 1 (Amara — Gold)
    (1001, 1,  2,  1, '2024-01-15', 2, 42.00, 84.00),
    (1002, 1,  1,  1, '2024-03-01', 1, 38.00, 38.00),
    (1003, 1,  8,  2, '2024-06-01', 3, 45.00, 135.00),
    (1004, 1,  13, 1, '2025-01-01', 2, 35.00, 70.00),
    (1005, 1,  3,  1, '2025-03-01', 1, 44.00, 44.00),
    -- Customer 2 (Siya — Silver)
    (1006, 2,  5,  2, '2024-02-01', 1, 36.00, 36.00),
    (1007, 2,  11, 3, '2024-04-15', 2, 42.00, 84.00),
    (1008, 2,  14, 2, '2024-08-01', 1, 30.00, 30.00),
    (1009, 2,  4,  2, '2025-02-01', 3, 28.00, 84.00),
    -- Customer 3 (Thandi — Bronze)
    (1010, 3,  12, 3, '2024-03-15', 1, 32.00, 32.00),
    (1011, 3,  14, 3, '2024-07-01', 2, 30.00, 60.00),
    -- Customer 4 (Lebo — Gold)
    (1012, 4,  9,  1, '2024-01-01', 2, 48.00, 96.00),
    (1013, 4,  2,  1, '2024-04-01', 4, 42.00, 168.00),
    (1014, 4,  13, 1, '2024-09-01', 1, 35.00, 35.00),
    (1015, 4,  3,  2, '2025-01-01', 3, 44.00, 132.00),
    (1016, 4,  8,  1, '2025-04-01', 2, 45.00, 90.00),
    -- Customer 5 (Kagiso — Bronze)
    (1017, 5,  4,  6, '2024-05-01', 1, 28.00, 28.00),
    (1018, 5,  14, 6, '2024-10-01', 1, 30.00, 30.00),
    -- Customer 6 (Naledi — Silver)
    (1019, 6,  10, 3, '2024-02-14', 2, 46.00, 92.00),
    (1020, 6,  11, 3, '2024-05-15', 1, 42.00, 42.00),
    (1021, 6,  15, 4, '2024-11-01', 2, 38.00, 76.00),
    (1022, 6,  2,  3, '2025-03-01', 1, 42.00, 42.00),
    -- Customer 7 (Sipho — Gold)
    (1023, 7,  8,  5, '2024-01-01', 3, 45.00, 135.00),
    (1024, 7,  9,  5, '2024-03-15', 2, 48.00, 96.00),
    (1025, 7,  2,  5, '2024-06-01', 4, 42.00, 168.00),
    (1026, 7,  13, 5, '2024-10-01', 2, 35.00, 70.00),
    (1027, 7,  3,  5, '2025-02-01', 3, 44.00, 132.00),
    (1028, 7,  1,  5, '2025-05-01', 2, 38.00, 76.00),
    -- Customer 8 (Ayanda — Bronze)
    (1029, 8,  12, 6, '2024-04-01', 1, 32.00, 32.00),
    -- Customer 9 (Tebogo — Silver)
    (1030, 9,  5,  1, '2024-02-01', 2, 36.00, 72.00),
    (1031, 9,  10, 2, '2024-05-01', 1, 46.00, 46.00),
    (1032, 9,  15, 1, '2024-09-01', 2, 38.00, 76.00),
    (1033, 9,  4,  2, '2025-01-01', 3, 28.00, 84.00),
    -- Customer 10 (Refilwe — Bronze)
    (1034, 10, 11, 4, '2024-06-01', 1, 42.00, 42.00),
    (1035, 10, 12, 4, '2024-12-01', 1, 32.00, 32.00),
    -- Customer 11 (Zanele — Gold)
    (1036, 11, 2,  3, '2024-01-15', 3, 42.00, 126.00),
    (1037, 11, 9,  3, '2024-04-01', 2, 48.00, 96.00),
    (1038, 11, 8,  3, '2024-07-01', 4, 45.00, 180.00),
    (1039, 11, 3,  4, '2024-10-01', 2, 44.00, 88.00),
    (1040, 11, 13, 3, '2025-01-01', 3, 35.00, 105.00),
    (1041, 11, 1,  3, '2025-04-01', 2, 38.00, 76.00),
    -- Customer 12 (Mpho — Silver)
    (1042, 12, 10, 4, '2024-03-01', 2, 46.00, 92.00),
    (1043, 12, 11, 4, '2024-08-01', 1, 42.00, 42.00),
    (1044, 12, 14, 5, '2025-03-01', 2, 30.00, 60.00),
    -- Customer 13 (Lindi — Bronze, no orders in 2025)
    (1045, 13, 12, 5, '2024-05-01', 1, 32.00, 32.00),
    (1046, 13, 14, 5, '2024-11-01', 1, 30.00, 30.00),
    -- Customer 14 (Bongani — Silver)
    (1047, 14, 5,  6, '2024-02-14', 2, 36.00, 72.00),
    (1048, 14, 15, 6, '2024-06-01', 2, 38.00, 76.00),
    (1049, 14, 4,  6, '2025-02-01', 4, 28.00, 112.00),
    -- Customer 15 (Ntombi — Gold)
    (1050, 15, 3,  3, '2024-01-01', 2, 44.00, 88.00),
    (1051, 15, 9,  3, '2024-04-15', 3, 48.00, 144.00),
    (1052, 15, 2,  4, '2024-08-01', 2, 42.00, 84.00),
    (1053, 15, 8,  3, '2025-01-01', 4, 45.00, 180.00),
    -- Customer 16 (Karabo — Bronze, no orders)
    -- Customer 17 (Dineo — Silver)
    (1054, 17, 10, 1, '2024-03-15', 1, 46.00, 46.00),
    (1055, 17, 11, 2, '2024-09-01', 2, 42.00, 84.00),
    (1056, 17, 5,  1, '2025-05-01', 1, 36.00, 36.00),
    -- Customer 18 (Lesego — Bronze)
    (1057, 18, 13, 6, '2024-07-01', 1, 35.00, 35.00),
    -- Customer 19 (Precious — Gold)
    (1058, 19, 8,  5, '2024-04-01', 2, 45.00, 90.00),
    (1059, 19, 2,  5, '2024-10-01', 3, 42.00, 126.00),
    (1060, 19, 9,  5, '2025-03-01', 2, 48.00, 96.00);
    -- Customer 20 (Tshepo — Silver, no orders yet)

SELECT COUNT(*) AS order_count FROM fact_orders;  -- expected: 60


In [0]:
-- ─────────────────────────────────────────────────────────────
-- STEP 8: Build customer_spend_summary helper table
-- Used heavily in window function examples
-- ─────────────────────────────────────────────────────────────
DROP TABLE IF EXISTS customer_spend_summary;

CREATE TABLE customer_spend_summary
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
AS
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    c.tier,
    COUNT(o.order_id)    AS total_orders,
    COALESCE(SUM(o.total_amount), 0)  AS total_spent
FROM  dim_customers  c
LEFT JOIN fact_orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name, c.tier;

SELECT * FROM customer_spend_summary
ORDER BY total_spent DESC;


In [0]:
-- ─────────────────────────────────────────────────────────────
-- STEP 9: Confirm all tables loaded correctly
-- ─────────────────────────────────────────────────────────────
SELECT 'dim_customers'        AS tbl, COUNT(*) AS rows FROM dim_customers
UNION ALL
SELECT 'dim_products',                COUNT(*) FROM dim_products
UNION ALL
SELECT 'dim_stores',                  COUNT(*) FROM dim_stores
UNION ALL
SELECT 'dim_date',                    COUNT(*) FROM dim_date
UNION ALL
SELECT 'fact_orders',                 COUNT(*) FROM fact_orders
UNION ALL
SELECT 'customer_spend_summary',      COUNT(*) FROM customer_spend_summary
ORDER BY tbl;

-- Expected:
-- customer_spend_summary : 20
-- dim_customers          : 20
-- dim_date               : 22
-- dim_orders             : 60
-- dim_products           : 16
-- dim_stores             :  6


### ✅ Setup complete — data notes

A few things to know about this dataset that are useful for teaching:

| Fact | Detail |
|------|--------|
| **No-order customers** | Customer 16 (Karabo) and Customer 20 (Tshepo) have never ordered — useful for NOT EXISTS |
| **Tiers** | 6 Gold · 6 Silver · 8 Bronze customers |
| **Inactive product** | Product 16 (Decaf Oat Latte) has `is_active = false` |
| **Decaf products** | Products 6 and 16 contain 'Decaf' — useful for NOT LIKE |
| **Regions** | North (3 stores), West (2 stores), East (1 store) — useful for region-based filters |
| **2025 orders** | Customers 1,2,4,6,7,9,11,12,14,15,17,19 have 2025 orders; others do not |
| **High-value orders** | Several orders exceed R150 — useful for subquery threshold examples |

You are now ready to work through the notebook sections.


## 1. Wildcards & Pattern Matching

Wildcards let you search for text patterns rather than exact values.
Use them when you don't know the full value, or when data is inconsistent.

| Wildcard | Meaning | Example |
|----------|---------|---------|
| `%` | Any sequence of zero or more characters | `LIKE 'Cap%'` matches Cape Town, Capitals |
| `_` | Exactly one character | `LIKE 'L__'` matches Lee, Lim, Luo |
| `ILIKE` | Case-insensitive LIKE (Databricks) | `ILIKE '%latte%'` matches Latte, LATTE, latte |
| `RLIKE` | Full regular expression | `RLIKE '[0-9]'` matches anything containing a digit |

> **LIKE vs ILIKE:** LIKE is case-sensitive. ILIKE is safer when your data has inconsistent casing.


In [0]:
-- WORKED EXAMPLE — LIKE, ILIKE, and NOT LIKE

-- All customers with a gmail address
SELECT customer_id, first_name, email
FROM   dim_customers
WHERE  email LIKE '%@gmail.com';


In [0]:
-- WORKED EXAMPLE — first name exactly 3 characters starting with L
SELECT customer_id, first_name
FROM   dim_customers
WHERE  first_name LIKE 'L__';
-- Two underscores = exactly 2 more characters after L


In [0]:
-- WORKED EXAMPLE — product name contains 'latte' in any casing
SELECT product_id, product_name, price
FROM   dim_products
WHERE  product_name ILIKE '%latte%';


In [0]:
-- WORKED EXAMPLE — products NOT containing 'Decaf', combined with OR
SELECT product_name, category
FROM   dim_products
WHERE  product_name NOT LIKE '%Decaf%'
  AND  (product_name ILIKE '%latte%' OR product_name ILIKE '%espresso%');


In [0]:
-- ✏️  YOUR TURN
-- Task 1a: Find all customers whose email ends with @yahoo.com.
-- Show customer_id, first_name, and email.

SELECT ...


In [0]:
-- ✏️  YOUR TURN
-- Task 1b: Find all products whose name starts with 'Oat'.
-- Show product_name and price.

SELECT ...


In [0]:
-- ✏️  YOUR TURN
-- Task 1c: Find all customers whose first name is exactly 5 characters long.
-- Show first_name and tier.
-- Hint: one underscore per character.

SELECT ...


In [0]:
-- ✏️  YOUR TURN
-- Task 1d (challenge): Find all products that contain a number in their name.
-- Use RLIKE.

SELECT ...


## 2. DDL — Data Definition Language

DDL commands define the *containers* that hold data — schemas, tables, columns.
DDL changes are usually permanent. Always think before running them.

### Key Databricks rules to remember
- `NOT NULL`, `DEFAULT`, and `PRIMARY KEY` go **inside** `CREATE TABLE`.
- `CHECK`, `UNIQUE`, and `FOREIGN KEY` must be added **after** creation with `ALTER TABLE`.
- `DEFAULT` values are **silently ignored** unless you set `TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')`.
- `RENAME COLUMN` and `DROP COLUMN` require column mapping to be enabled first.


In [0]:
-- WORKED EXAMPLE — Create a schema and activate it
CREATE SCHEMA IF NOT EXISTS my_practice
    COMMENT 'My practice schema';

USE my_practice;
SELECT current_database();


In [0]:
-- WORKED EXAMPLE — CREATE TABLE with NOT NULL, DEFAULT, PRIMARY KEY, TBLPROPERTIES
CREATE TABLE IF NOT EXISTS dim_customers (
    customer_id    BIGINT  NOT NULL,
    first_name     STRING  NOT NULL,
    last_name      STRING  NOT NULL,
    email          STRING,
    tier           STRING  DEFAULT 'Bronze',    -- needs TBLPROPERTIES below
    loyalty_points INT     DEFAULT 0,           -- needs TBLPROPERTIES below
    created_at     DATE,
    CONSTRAINT pk_customer PRIMARY KEY (customer_id)
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');
-- ^ Without this line, DEFAULT 'Bronze' and DEFAULT 0 are silently ignored


In [0]:
-- WORKED EXAMPLE — Add UNIQUE and CHECK constraints AFTER creation
-- UNIQUE and CHECK cannot go inside CREATE TABLE in Databricks

ALTER TABLE dim_customers
    ADD CONSTRAINT uq_email UNIQUE (email);

ALTER TABLE dim_customers
    ADD CONSTRAINT chk_tier CHECK (tier IN ('Bronze', 'Silver', 'Gold'));

-- Enforce the CHECK (informational by default)
ALTER TABLE dim_customers
    ALTER CONSTRAINT chk_tier ENFORCED;


In [0]:
-- WORKED EXAMPLE — Enable column mapping, then rename and drop a column
-- Must set all three properties together or RENAME/DROP will fail
ALTER TABLE dim_customers
    SET TBLPROPERTIES (
        'delta.columnMapping.mode' = 'name',
        'delta.minReaderVersion'   = '2',
        'delta.minWriterVersion'   = '5'
    );

ALTER TABLE dim_customers ADD COLUMN phone STRING;
ALTER TABLE dim_customers RENAME COLUMN phone TO phone_number;
ALTER TABLE dim_customers DROP COLUMN phone_number;


In [0]:
-- WORKED EXAMPLE — Backup a table, time travel, and view history
-- Backup before any destructive operation
CREATE TABLE dim_customers_backup AS
SELECT * FROM dim_customers;

-- Query a previous version
SELECT * FROM dim_customers VERSION AS OF 0;

-- See the full transaction log
DESCRIBE HISTORY dim_customers;


In [0]:
-- ✏️  YOUR TURN
-- Task 2a: Create your own schema called my_coffee_shop.
-- Activate it and confirm you are in it.

CREATE SCHEMA ...


In [0]:
-- ✏️  YOUR TURN
-- Task 2b: Inside my_coffee_shop, create a table called dim_products with:
--   - product_id  BIGINT NOT NULL
--   - product_code STRING NOT NULL
--   - product_name STRING NOT NULL
--   - price DECIMAL(8,2) NOT NULL
--   - category STRING DEFAULT 'Uncategorized'
--   - is_active BOOLEAN DEFAULT true
--   - A named PRIMARY KEY on product_id
--   - The correct TBLPROPERTIES for DEFAULT values to work

CREATE TABLE IF NOT EXISTS dim_products (
    ...
)
USING DELTA
TBLPROPERTIES (...);


In [0]:
-- ✏️  YOUR TURN
-- Task 2c: After creating dim_products, add:
--   - A CHECK constraint ensuring price > 0
--   - A UNIQUE constraint on product_code

ALTER TABLE dim_products ...


In [0]:
-- ✏️  YOUR TURN
-- Task 2d: Create a fact_orders table with the following columns:
--   order_id, customer_id, product_id, store_id (BIGINT NOT NULL)
--   order_date (DATE NOT NULL)
--   quantity (INT NOT NULL)
--   unit_price, total_amount (DECIMAL(10,2) NOT NULL)
--   Named PRIMARY KEY on order_id
--   Partitioned by order_date

CREATE TABLE IF NOT EXISTS fact_orders (
    ...
)
USING DELTA
PARTITIONED BY (...);


## 3. DML — Data Manipulation Language

DML commands work with the *data inside* your tables.

**The most important rule:** always run a `SELECT` with the same `WHERE` clause
before running `UPDATE` or `DELETE`. If the SELECT returns the rows you expect,
then run the DML.

| Command | What it does |
|---------|-------------|
| `INSERT INTO … VALUES` | Add specific rows |
| `INSERT INTO … SELECT` | Copy rows from a query |
| `UPDATE … SET … WHERE` | Modify matching rows |
| `DELETE FROM … WHERE` | Remove matching rows |
| `CREATE TABLE … AS SELECT` (CTAS) | Create and fill a table in one step |


In [0]:
-- WORKED EXAMPLE — Single and multi-row INSERT
INSERT INTO dim_customers
    (customer_id, first_name, last_name, email, tier, loyalty_points, created_at)
VALUES
    (1, 'Amara',  'Nkosi',   'amara@gmail.com',  'Gold',   500, '2023-01-15'),
    (2, 'Siya',   'Dlamini', 'siya@yahoo.com',   'Silver', 200, '2023-03-10'),
    (3, 'Thandi', 'Mokoena', 'thandi@gmail.com', 'Bronze',  50, '2023-06-01');


In [0]:
-- WORKED EXAMPLE — UPDATE: always SELECT first to check who is affected
-- Step 1: verify before touching data
SELECT * FROM dim_customers WHERE tier = 'Bronze';

-- Step 2: run the UPDATE
UPDATE dim_customers
SET    tier           = 'Silver',
       loyalty_points = loyalty_points + 50
WHERE  tier = 'Bronze';


In [0]:
-- WORKED EXAMPLE — DELETE: verify the row, then delete
-- Step 1: confirm which row will be deleted
SELECT * FROM dim_customers WHERE customer_id = 3;

-- Step 2: delete
DELETE FROM dim_customers WHERE customer_id = 3;


In [0]:
-- WORKED EXAMPLE — INSERT INTO SELECT: copy Gold customers to a new table
-- Create the destination table first (empty copy via CTAS)
CREATE TABLE vip_customers AS
SELECT * FROM dim_customers WHERE 1=0;

-- Copy Gold-tier customers in
INSERT INTO vip_customers
    (customer_id, first_name, last_name, email, tier, loyalty_points, created_at)
SELECT customer_id, first_name, last_name, email, tier, loyalty_points, created_at
FROM   dim_customers
WHERE  tier = 'Gold';

SELECT * FROM vip_customers;


In [0]:
-- WORKED EXAMPLE — CTAS: create a summary table in one step
CREATE TABLE customer_spend_summary AS
SELECT
    c.customer_id,
    c.first_name,
    c.tier,
    COUNT(o.order_id)    AS total_orders,
    SUM(o.total_amount)  AS total_spent
FROM  dim_customers  c
LEFT JOIN fact_orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.first_name, c.tier;


In [0]:
-- ✏️  YOUR TURN
-- Task 3a: Insert at least 5 products into dim_products.
-- Include at least 2 different categories.
-- Use a single multi-row INSERT.

INSERT INTO dim_products
    (product_id, product_code, product_name, price, category, is_active)
VALUES
    (...),
    (...);


In [0]:
-- ✏️  YOUR TURN
-- Task 3b: All products in the 'Coffee' category are getting a 10% price increase.
-- First SELECT to confirm which products will be affected.
-- Then run the UPDATE.

-- Step 1: confirm
SELECT ...

-- Step 2: update
UPDATE dim_products
SET    price = ...
WHERE  ...;


In [0]:
-- ✏️  YOUR TURN
-- Task 3c: Create a product_summary table using CTAS.
-- It should show: category, number of products, and average price per category.

CREATE TABLE product_summary AS
SELECT ...


## 4. Window Functions

Window functions calculate across a set of rows **related to the current row**,
without collapsing the result like `GROUP BY` does.
Every row stays in the result — you just get extra calculated columns.

```sql
function_name(column) OVER (
    PARTITION BY partition_column   -- split into groups
    ORDER BY     order_column       -- order within each group
)
```

### Categories
| Category | Functions | Use for |
|----------|-----------|---------|
| Ranking | `ROW_NUMBER`, `RANK`, `DENSE_RANK`, `NTILE` | Top-N per group, percentile bands |
| Aggregation | `SUM`, `AVG`, `COUNT`, `MIN`, `MAX` with `OVER()` | Running totals, % of total |
| Offset | `LAG`, `LEAD` | Period-over-period comparisons |
| Boundary | `FIRST_VALUE`, `LAST_VALUE` | Earliest/latest per group |


### 4a. Ranking Functions

**ROW_NUMBER** — always unique, no ties.
**RANK** — ties get the same rank, next rank skips (1, 1, 3).
**DENSE_RANK** — ties get the same rank, no gap (1, 1, 2).
**NTILE(n)** — divides rows into n equal buckets.


In [0]:
-- WORKED EXAMPLE — ROW_NUMBER, RANK, DENSE_RANK, NTILE per tier
SELECT
    first_name,
    tier,
    total_spent,
    ROW_NUMBER() OVER (PARTITION BY tier ORDER BY total_spent DESC)  AS row_num,
    RANK()       OVER (PARTITION BY tier ORDER BY total_spent DESC)  AS rnk,
    DENSE_RANK() OVER (PARTITION BY tier ORDER BY total_spent DESC)  AS dense_rnk,
    NTILE(4)     OVER (PARTITION BY tier ORDER BY total_spent DESC)  AS quartile
FROM  customer_spend_summary
ORDER BY tier, total_spent DESC;


In [0]:
-- ✏️  YOUR TURN
-- Task 4a: Rank all products within each category by price (highest first).
-- Use DENSE_RANK. Show product_name, category, price, and the rank.

SELECT
    product_name,
    category,
    price,
    DENSE_RANK() OVER (...) AS price_rank
FROM dim_products
ORDER BY category, price DESC;


### 4b. Aggregate Window Functions

These are the standard aggregation functions (`SUM`, `AVG`, etc.) with an `OVER()` clause.
They calculate across a window without collapsing rows.

**Running total** — `ORDER BY` without `PARTITION BY` accumulates across the whole table.
**Customer total** — `PARTITION BY customer_id` resets for each customer.
**% of total** — divide the row amount by its partition total.


In [0]:
-- WORKED EXAMPLE — Running total, customer total, tier average, % of customer total
SELECT
    o.order_id,
    c.first_name,
    c.tier,
    o.order_date,
    o.total_amount,
    SUM(o.total_amount) OVER (ORDER BY o.order_date)
                                                          AS running_total,
    SUM(o.total_amount) OVER (PARTITION BY o.customer_id)
                                                          AS customer_total,
    AVG(o.total_amount) OVER (PARTITION BY c.tier)
                                                          AS tier_avg,
    ROUND(
        o.total_amount /
        SUM(o.total_amount) OVER (PARTITION BY o.customer_id) * 100
    , 2)                                                   AS pct_of_customer_total
FROM  fact_orders    o
JOIN  dim_customers  c  ON o.customer_id = c.customer_id
ORDER BY o.order_date;


In [0]:
-- ✏️  YOUR TURN
-- Task 4b: For each store, calculate:
--   - The total revenue for that store (SUM OVER PARTITION BY store_id)
--   - Each order's amount as a % of its store's total revenue (round to 2 dp)
-- Join fact_orders to dim_stores. Show store_name, order_id, total_amount,
-- store_total, and pct_of_store.

SELECT
    s.store_name,
    o.order_id,
    o.total_amount,
    SUM(...) OVER (PARTITION BY ...) AS store_total,
    ROUND(...)                       AS pct_of_store
FROM fact_orders   o
JOIN dim_stores    s ON o.store_id = s.store_id
ORDER BY s.store_name, o.order_date;


### 4c. LAG and LEAD

`LAG` looks **back** to a previous row. `LEAD` looks **forward** to a next row.
Both work within the partition defined by `PARTITION BY`.

```sql
LAG(column, offset, default_if_no_row) OVER (PARTITION BY ... ORDER BY ...)
```

Use them for period-over-period comparisons — e.g. how did this order compare to the customer's last?


In [0]:
-- WORKED EXAMPLE — LAG, LEAD, and change vs previous order
SELECT
    o.order_id,
    c.first_name,
    o.order_date,
    o.total_amount,
    LAG(o.total_amount,  1, 0) OVER (PARTITION BY o.customer_id
                                     ORDER BY o.order_date)  AS prev_amount,
    LEAD(o.total_amount, 1, 0) OVER (PARTITION BY o.customer_id
                                     ORDER BY o.order_date)  AS next_amount,
    o.total_amount - LAG(o.total_amount, 1, 0)
                     OVER (PARTITION BY o.customer_id
                           ORDER BY o.order_date)            AS change_vs_prev
FROM  fact_orders    o
JOIN  dim_customers  c  ON o.customer_id = c.customer_id
ORDER BY c.first_name, o.order_date;


In [0]:
-- ✏️  YOUR TURN
-- Task 4c: For each product, find the previous order's unit_price using LAG
-- and the next order's unit_price using LEAD (within the same product).
-- Show order_id, product_id, order_date, unit_price, prev_price, next_price.

SELECT
    o.order_id,
    o.product_id,
    o.order_date,
    o.unit_price,
    LAG(...)  OVER (...) AS prev_price,
    LEAD(...) OVER (...) AS next_price
FROM fact_orders o
ORDER BY o.product_id, o.order_date;


### 4d. FIRST_VALUE and LAST_VALUE

`FIRST_VALUE` returns the value from the **first row** in the window.
`LAST_VALUE` returns the value from the **last row** in the window.

Use them to show a reference point alongside every row — e.g.
each customer's first order date next to every subsequent order.


In [0]:
-- WORKED EXAMPLE — FIRST_VALUE: each order alongside the customer's first order date
SELECT
    o.order_id,
    c.first_name,
    o.order_date,
    FIRST_VALUE(o.order_date) OVER (
        PARTITION BY o.customer_id
        ORDER BY o.order_date
    )  AS first_order_date,
    DATEDIFF(o.order_date,
             FIRST_VALUE(o.order_date) OVER (
                 PARTITION BY o.customer_id
                 ORDER BY o.order_date
             )
    )  AS days_since_first
FROM  fact_orders    o
JOIN  dim_customers  c  ON o.customer_id = c.customer_id
ORDER BY c.first_name, o.order_date;


In [0]:
-- ✏️  YOUR TURN
-- Task 4d: For each customer, show their most expensive order ever (FIRST_VALUE
-- ordered by total_amount DESC) alongside every order row.
-- Show customer first_name, order_id, total_amount, and max_order_ever.

SELECT
    c.first_name,
    o.order_id,
    o.total_amount,
    FIRST_VALUE(o.total_amount) OVER (
        PARTITION BY o.customer_id
        ORDER BY o.total_amount DESC
    )  AS max_order_ever
FROM fact_orders   o
JOIN dim_customers c ON o.customer_id = c.customer_id
ORDER BY c.first_name;


## 5. Advanced Querying

This section covers tools that let you build multi-step logic inside a single query.

| Tool | What it is | When to use |
|------|-----------|-------------|
| **Subquery** | A SELECT inside another SELECT | When you need a calculated value or list as a filter |
| **CTE** | A named temporary result set (`WITH name AS (...)`) | When you have nested logic or need the same result twice |
| **EXISTS / NOT EXISTS** | A yes/no check — does a matching row exist? | Checking presence or absence efficiently |
| **ANY / ALL** | Compare against a whole result set | When you need 'beats at least one' or 'beats everyone' |


### 5a. Subqueries

The **inner query runs first**. Its result is handed to the outer query.

**Non-correlated** — inner runs once, returns a value or list.
**Correlated** — inner references the outer row, runs once per row (slower).


In [0]:
-- WORKED EXAMPLE — Non-correlated subquery: customers who spent above average
SELECT customer_id, first_name, total_spent
FROM   customer_spend_summary
WHERE  total_spent > (
    SELECT AVG(total_spent)       -- runs ONCE, returns e.g. 4250
    FROM   customer_spend_summary
)
ORDER BY total_spent DESC;


In [0]:
-- WORKED EXAMPLE — Subquery returning a list: customers who ordered this year
SELECT customer_id, first_name, tier
FROM   dim_customers
WHERE  customer_id IN (
    SELECT DISTINCT customer_id
    FROM   fact_orders
    WHERE  order_date >= '2025-01-01'
);


In [0]:
-- WORKED EXAMPLE — Correlated subquery: customers whose latest order was in 2025
SELECT c.customer_id, c.first_name
FROM   dim_customers c
WHERE (
    SELECT MAX(o.order_date)
    FROM   fact_orders o
    WHERE  o.customer_id = c.customer_id   -- links to outer row
) >= '2025-01-01';
-- Inner runs once PER customer row


In [0]:
-- ✏️  YOUR TURN
-- Task 5a: Write a non-correlated subquery that returns all orders
-- where unit_price is greater than the overall average unit_price.
-- Show order_id, customer_id, unit_price, and total_amount.

SELECT order_id, customer_id, unit_price, total_amount
FROM   fact_orders
WHERE  unit_price > (
    SELECT ...
);


In [0]:
-- ✏️  YOUR TURN
-- Task 5b: Using NOT IN with a subquery, find all customers who did NOT
-- place any order in 2025.
-- Show customer_id, first_name, and tier.

SELECT customer_id, first_name, tier
FROM   dim_customers
WHERE  customer_id NOT IN (
    SELECT ...
);


### 5b. CTEs — Common Table Expressions

A CTE gives a **name** to a temporary result set defined at the top of the query with `WITH`.
It only lives for that one query. Use CTEs when subqueries would nest two or more levels deep.

```sql
WITH cte_name AS (
    SELECT ...    -- define the named result
)
SELECT * FROM cte_name WHERE ...;   -- use it like a table
```

You can **chain multiple CTEs** by separating them with a comma.
Each CTE can reference the ones defined before it.


In [0]:
-- WORKED EXAMPLE — Single CTE: average spend per tier, only where avg > 3000
WITH tier_averages AS (
    SELECT
        tier,
        AVG(total_spent)  AS avg_spent
    FROM  customer_spend_summary
    GROUP BY tier
)
SELECT  tier, avg_spent
FROM    tier_averages
WHERE   avg_spent > 3000
ORDER BY avg_spent DESC;


In [0]:
-- WORKED EXAMPLE — Chained CTEs: customers above their own tier average
WITH
-- Step 1: calculate average spend per tier
tier_summary AS (
    SELECT  tier, AVG(total_spent) AS avg_spent
    FROM    customer_spend_summary
    GROUP BY tier
),
-- Step 2: find customers who beat their tier average
above_average AS (
    SELECT
        c.first_name,
        c.tier,
        c.total_spent,
        t.avg_spent,
        c.total_spent - t.avg_spent  AS above_by
    FROM    customer_spend_summary  c
    JOIN    tier_summary            t  ON c.tier = t.tier
    WHERE   c.total_spent > t.avg_spent
)
SELECT * FROM above_average
ORDER BY above_by DESC
LIMIT  10;


In [0]:
-- ✏️  YOUR TURN
-- Task 5c: Using a single CTE, calculate total revenue per store.
-- In the final SELECT, return only stores with revenue above 50 000.
-- Show store_id, store_name, and total_revenue.

WITH store_revenue AS (
    SELECT
        s.store_id,
        s.store_name,
        SUM(o.total_amount)  AS total_revenue
    FROM  ...
    GROUP BY ...
)
SELECT ...
FROM   store_revenue
WHERE  ...;


In [0]:
-- ✏️  YOUR TURN
-- Task 5d: Chain two CTEs:
--   CTE 1 (daily_revenue): total revenue per day from fact_orders
--   CTE 2 (high_days): only days where revenue > 5000
-- Final SELECT: top 5 highest-revenue days

WITH
daily_revenue AS (
    SELECT order_date, SUM(total_amount) AS revenue
    FROM   ...
    GROUP BY ...
),
high_days AS (
    SELECT ...
    FROM   daily_revenue
    WHERE  ...
)
SELECT ...
FROM   high_days
ORDER BY ...
LIMIT 5;


In [0]:
-- ✏️  YOUR TURN
-- Task 5e (challenge): Combine a CTE and a window function.
-- CTE: calculate each customer's average order amount.
-- Final SELECT: join back to fact_orders, add a CASE column labelling
-- each order as 'Above Average' or 'Below Average' vs that customer's average.

WITH customer_avg AS (
    SELECT customer_id, AVG(total_amount) AS avg_order
    FROM   ...
    GROUP BY ...
)
SELECT
    o.order_id,
    o.customer_id,
    o.total_amount,
    ca.avg_order,
    CASE WHEN ... THEN 'Above Average' ELSE 'Below Average' END AS vs_average
FROM  fact_orders   o
JOIN  customer_avg  ca ON ...
ORDER BY o.customer_id, o.order_date;


### 5c. EXISTS, NOT EXISTS, ANY, ALL

**EXISTS** asks a yes/no question: does this subquery return *at least one row*?
It stops as soon as it finds a match — faster than `IN` on large tables.
Always write `SELECT 1` inside EXISTS — you don't need any column values.

**NOT EXISTS** is the opposite: true when zero rows are returned.
Prefer it over `NOT IN` — `NOT IN` behaves unexpectedly when the subquery returns a NULL.

**ANY** — true if the condition holds for *at least one* value in the subquery result.
**ALL** — true only if the condition holds for *every* value.


In [0]:
-- WORKED EXAMPLE — EXISTS: customers who placed at least one order
SELECT c.customer_id, c.first_name, c.tier
FROM   dim_customers c
WHERE  EXISTS (
    SELECT 1
    FROM   fact_orders o
    WHERE  o.customer_id = c.customer_id
)
ORDER BY c.tier;


In [0]:
-- WORKED EXAMPLE — NOT EXISTS: customers who never ordered
SELECT c.customer_id, c.first_name, c.tier
FROM   dim_customers c
WHERE  NOT EXISTS (
    SELECT 1
    FROM   fact_orders o
    WHERE  o.customer_id = c.customer_id
);


In [0]:
-- WORKED EXAMPLE — ANY and ALL: products vs Gold-tier order prices
-- ANY: price < at least one Gold purchase (less than the minimum)
SELECT product_name, price
FROM   dim_products
WHERE  price < ANY (
    SELECT o.unit_price
    FROM   fact_orders    o
    JOIN   dim_customers  c  ON o.customer_id = c.customer_id
    WHERE  c.tier = 'Gold'
);

-- ALL: price < every Gold purchase (less than the maximum)
SELECT product_name, price
FROM   dim_products
WHERE  price < ALL (
    SELECT o.unit_price
    FROM   fact_orders    o
    JOIN   dim_customers  c  ON o.customer_id = c.customer_id
    WHERE  c.tier = 'Gold'
);


In [0]:
-- ✏️  YOUR TURN
-- Task 5f: Using EXISTS, return all customers who placed at least one order
-- with a total_amount greater than 500.
-- Show customer_id, first_name, and tier.

SELECT c.customer_id, c.first_name, c.tier
FROM   dim_customers c
WHERE  EXISTS (
    SELECT 1
    FROM   fact_orders o
    WHERE  o.customer_id = c.customer_id
      AND  ...
);


In [0]:
-- ✏️  YOUR TURN
-- Task 5g: Using NOT EXISTS, find customers who have never ordered
-- from a store in the 'West' region.
-- Join fact_orders to dim_stores inside the subquery.

SELECT c.customer_id, c.first_name
FROM   dim_customers c
WHERE  NOT EXISTS (
    SELECT 1
    FROM   fact_orders  o
    JOIN   dim_stores   s  ON o.store_id = s.store_id
    WHERE  o.customer_id = c.customer_id
      AND  s.region = 'West'
);


## 6. Views

A **view** is a saved SELECT query you can reference by name like a table.
It stores **no data** — every query against a view reruns the underlying SELECT live.

| Feature | Regular View | Temporary View | CTAS Table |
|---------|-------------|----------------|------------|
| Stores data? | No | No | Yes |
| Persists after session? | Yes | No | Yes |
| Visible in catalog? | Yes | No | Yes |
| Best for | Reusable logic, access control | Intermediate notebook steps | Performance-critical results |

> **When to use a view vs CTAS:** If the result is queried once or twice, a view is fine.
> If it's queried hundreds of times by many users, materialise it with CTAS.


In [0]:
-- WORKED EXAMPLE — Create a view joining customers and orders
CREATE VIEW vw_customer_order_summary AS
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    c.tier,
    COUNT(o.order_id)    AS total_orders,
    SUM(o.total_amount)  AS total_spent,
    MAX(o.order_date)    AS last_order_date
FROM  dim_customers  c
LEFT JOIN fact_orders o  ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name, c.tier;


In [0]:
-- WORKED EXAMPLE — Query the view like a table
-- Filter and sort the view — it runs the underlying SELECT live
SELECT *
FROM   vw_customer_order_summary
WHERE  tier = 'Gold'
ORDER BY total_spent DESC;


In [0]:
-- WORKED EXAMPLE — Replace, list, and drop views
-- Update the view definition
CREATE OR REPLACE VIEW vw_customer_order_summary AS
SELECT ... ;   -- new definition goes here

-- Session-only temporary view
CREATE OR REPLACE TEMPORARY VIEW temp_bronze AS
SELECT * FROM dim_customers WHERE tier = 'Bronze';

-- List and remove
SHOW VIEWS;
DROP VIEW IF EXISTS vw_customer_order_summary;


In [0]:
-- ✏️  YOUR TURN
-- Task 6a: Create a view called vw_product_revenue that shows:
--   product_id, product_name, category,
--   total units sold (SUM of quantity), and total revenue (SUM of total_amount).
-- Join dim_products to fact_orders.

CREATE VIEW vw_product_revenue AS
SELECT
    p.product_id,
    p.product_name,
    p.category,
    SUM(o.quantity)     AS units_sold,
    SUM(o.total_amount) AS total_revenue
FROM  dim_products  p
JOIN  fact_orders   o  ON p.product_id = o.product_id
GROUP BY ...;


In [0]:
-- ✏️  YOUR TURN
-- Task 6b: Query vw_product_revenue to find the top 3 products by total_revenue.

SELECT ...
FROM   vw_product_revenue
ORDER BY ...
LIMIT 3;


In [0]:
-- ✏️  YOUR TURN
-- Task 6c: Create a TEMPORARY view called temp_recent_orders containing
-- all orders from the last 90 days (use CURRENT_DATE).
-- Then query it to count how many recent orders each customer placed.

CREATE OR REPLACE TEMPORARY VIEW temp_recent_orders AS
SELECT *
FROM   fact_orders
WHERE  order_date >= ...

-- Now query it
SELECT customer_id, COUNT(*) AS recent_order_count
FROM   temp_recent_orders
GROUP BY customer_id
ORDER BY recent_order_count DESC;


In [0]:
-- ============================================================
-- SECTION 7 — STORED PROCEDURES
-- ============================================================


## 7. Stored Procedures

A **stored procedure** is a named block of SQL logic saved in the database.
You execute it with `CALL`. Unlike a view, it can contain:
- `INSERT`, `UPDATE`, `DELETE`
- `IF / ELSE` conditional logic
- Input parameters
- Error handling with `SIGNAL`

```sql
CREATE OR REPLACE PROCEDURE procedure_name(
    IN param_name  DATA_TYPE
)
BEGIN
    -- validation
    IF condition THEN
        SIGNAL SQLSTATE '45000' SET MESSAGE_TEXT = 'error message';
    END IF;

    -- business logic
    UPDATE ...;
END;
```

> **Always validate first.** `SIGNAL` stops execution immediately.
> The DML only runs if all validation passes.


In [0]:
-- WORKED EXAMPLE — Procedure to upgrade a customer's tier
CREATE OR REPLACE PROCEDURE upgrade_customer_tier(
    IN p_customer_id  BIGINT,
    IN p_new_tier     STRING
)
BEGIN
    -- Validate the tier value before touching any data
    IF p_new_tier NOT IN ('Bronze', 'Silver', 'Gold') THEN
        SIGNAL SQLSTATE '45000'
            SET MESSAGE_TEXT = 'Invalid tier. Must be Bronze, Silver, or Gold.';
    END IF;

    -- Business logic
    UPDATE dim_customers
    SET    tier = p_new_tier
    WHERE  customer_id = p_customer_id;
END;


In [0]:
-- WORKED EXAMPLE — Call the procedure with a valid and an invalid value
-- Valid call
CALL upgrade_customer_tier(2, 'Gold');

-- Invalid call — should raise an error
CALL upgrade_customer_tier(2, 'Platinum');
-- Paste the error message as a comment below:
-- Error: ...


In [0]:
-- WORKED EXAMPLE — List and drop a procedure
SHOW PROCEDURES;
DROP PROCEDURE IF EXISTS upgrade_customer_tier;


In [0]:
-- ✏️  YOUR TURN
-- Task 7a: Create a procedure called add_loyalty_points that:
--   - Accepts p_customer_id (BIGINT) and p_points_to_add (INT)
--   - Validates that p_points_to_add is greater than zero
--   - Adds the points to the customer's loyalty_points balance

CREATE OR REPLACE PROCEDURE add_loyalty_points(
    IN p_customer_id   BIGINT,
    IN p_points_to_add INT
)
BEGIN
    -- Validate
    IF ... THEN
        SIGNAL SQLSTATE '45000'
            SET MESSAGE_TEXT = '...';
    END IF;

    -- Update
    UPDATE dim_customers
    SET    loyalty_points = loyalty_points + ...
    WHERE  customer_id = ...;
END;


In [0]:
-- ✏️  YOUR TURN
-- Task 7b: Call add_loyalty_points to add 100 points to customer_id 1.
-- Then call it with -50 points and paste the error message as a comment.

CALL add_loyalty_points(1, 100);

CALL add_loyalty_points(1, -50);
-- Error: ...


In [0]:
-- ✏️  YOUR TURN
-- Task 7c (challenge): Create a procedure called deactivate_product that:
--   - Accepts p_product_id (BIGINT)
--   - Checks the product exists (use EXISTS in an IF condition)
--   - Raises an error if the product is not found
--   - Sets is_active = false if it does exist

CREATE OR REPLACE PROCEDURE deactivate_product(
    IN p_product_id BIGINT
)
BEGIN
    IF NOT EXISTS (
        SELECT 1 FROM dim_products WHERE product_id = p_product_id
    ) THEN
        SIGNAL SQLSTATE '45000'
            SET MESSAGE_TEXT = 'Product not found.';
    END IF;

    UPDATE dim_products
    SET    is_active = false
    WHERE  product_id = p_product_id;
END;


##  Well done!

You have worked through all seven sections of the Advanced SQL practical.

**Before you submit:**
- [ ] All YOUR TURN cells have your own SQL written in them
- [ ] All cells have been run — results are visible below each cell
- [ ] Error messages from intentional failure tests are pasted as comments

**Topics covered in this notebook:**

| Section | Topic |
|---------|-------|
| 1 | Wildcards & Pattern Matching |
| 2 | DDL — Building the Schema |
| 3 | DML — Populating & Modifying Data |
| 4 | Window Functions |
| 5 | Advanced Querying — Subqueries, CTEs, EXISTS, ANY/ALL |
| 6 | Views |
| 7 | Stored Procedures |

*BrightLearn Advanced SQL — Advanced Data Analytics*
*The best way to learn SQL is to write SQL.*
